# ML Study Tracker 4: Expanded Practical ML Use Cases

This notebook provides end-to-end, production-grade implementation pipelines for the complete classical machine learning registry. Each workflow is self-contained, leveraging robust data preprocessing via `ColumnTransformer`, structured training routines within `Pipeline` wrappers, and distinct validation card metrics.

## 📋 Table of Contents
- [Common Imports & Configuration](#Common-Imports-&-Configuration)
- [1. Linear Regression for House Price Prediction](#1.-Linear-Regression-for-House-Price-Prediction)
- [2. Logistic Regression for Customer Churn Prediction](#2.-Logistic-Regression-for-Customer-Churn-Prediction)
- [3. Time Series Forecasting for Sales/Stock Prediction](#3.-Time-Series-Forecasting-for-Sales/Stock-Prediction)
- [4. Random Forest Classifier for Credit Risk/Scoring](#4.-Random-Forest-Classifier-for-Credit-Risk/Scoring)
- [5. Support Vector Machine (SVM) + PCA for High-Dimensional Data](#5.-Support-Vector-Machine-(SVM)-+-PCA-for-High-Dimensional-Data)
- [6. K-Means Clustering for Customer Segmentation](#6.-K-Means-Clustering-for-Customer-Segmentation)
- [7. Multinomial Naive Bayes for Text Classification](#7.-Multinomial-Naive-Bayes-for-Text-Classification)
- [Expanded Mini Project Tracker](#Expanded-Mini-Project-Tracker)
- [Expanded Experiment Log](#Expanded-Experiment-Log)

In [ ]:
# Common Imports & Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    silhouette_score, classification_report, confusion_matrix
)

# Set random seed for reproducibility across all use cases
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Linear Regression for House Price Prediction
**Dataset Target:** California Housing Dataset (Popular Public Benchmark)

### Study Checklist
- [ ] Data Exploration & Target Skew Check
- [ ] Feature Scaling
- [ ] Model Training (OLS vs. Regularized Ridge/Lasso)
- [ ] Evaluation Metrics (RMSE, MAE, R^2)
- [ ] Residual Analysis (Homoscedasticity check)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: House Price Prediction
from sklearn.linear_model import Ridge
from sklearn.datasets import fetch_california_housing

# 1. Fetch Popular Public Data Source
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# 2. Setup Features and Target
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# 4. Pipeline Construction (Scale -> Regularized Regression)
house_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# 5. Fit & Evaluate
house_pipeline.fit(X_train, y_train)
y_pred = house_pipeline.predict(X_test)

print("--- House Price Prediction Metrics ---")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R^2:  {r2_score(y_test, y_pred):.4f}")

# 6. Residual Verification
residuals = y_test - y_pred
print(f"Mean of Residuals (Want ~0): {residuals.mean():.4f}")

## 2. Logistic Regression for Customer Churn Prediction
**Dataset Target:** Imbalanced Customer Behavior/Churn Analysis

### Study Checklist
- [ ] Imbalance Check & Evaluation Metric Selection (Avoid raw Accuracy)
- [ ] Dummy Encoding / One-Hot Encoding for Categories
- [ ] Addressing Imbalance (Class weights vs. Resampling)
- [ ] Extracting Decision Probabilities & Threshold Tuning
- [ ] Coefficient Odds-Ratio Interpretation

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Customer Churn Prediction
from sklearn.linear_model import LogisticRegression

# 1. Create a Synthetic Messy Churn Dataset to Simulate Production Data
n_samples = 1000
raw_data = pd.DataFrame({
    'Age': np.random.randint(18, 70, n_samples),
    'Tenure_Months': np.random.randint(0, 72, n_samples),
    'Contract_Type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'Monthly_Charges': np.random.uniform(20, 120, n_samples)
})

# Construct ground truth dependent on categorical and numeric patterns
churn_logit = (-2.0 
               + (raw_data['Contract_Type'] == 'Month-to-month') * 1.8 
               + raw_data['Monthly_Charges'] * 0.01 
               - raw_data['Tenure_Months'] * 0.03)
churn_prob = 1 / (1 + np.exp(-churn_logit))
raw_data['Churn'] = (np.random.rand(n_samples) < churn_prob).astype(int)

# 2. Identify Features and Target
numeric_features = ['Age', 'Tenure_Months', 'Monthly_Charges']
categorical_features = ['Contract_Type']

X = raw_data.drop(columns='Churn')
y = raw_data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

# 3. Preprocessing Transformer Block
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

# 4. Balanced Logistic Regression Pipeline
churn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED))
])

# 5. Fit, Predict Proba, and Evaluate via F1-Score
churn_pipeline.fit(X_train, y_train)
y_pred_proba = churn_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("--- Customer Churn Prediction Metrics ---")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

## 3. Time Series Forecasting for Sales/Stock Prediction
**Dataset Target:** Supermarket Sales / Historical Demand Sequences

### Study Checklist
- [ ] Structural Decomposition (Trend, Seasonality, Residual Noise)
- [ ] Designing Lag Features without Future Data Leakage
- [ ] Adding Rolling Windows (Mean, Std)
- [ ] Executing Chronological Train/Test Time Split
- [ ] Autoregressive ML Evaluation Benchmark

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Time Series Forecasting
from sklearn.linear_model import LinearRegression

# 1. Generate Synthetic Sequential Sales Time Series
t = np.arange(120)
trend = 0.3 * t
seasonality = 10 * np.sin(2 * np.pi * t / 7)  # Weekly cycle
noise = np.random.normal(0, 1.5, 120)
sales_series = pd.DataFrame({'Sales': 100 + trend + seasonality + noise})

# 2. Engineer Lag and Rolling Statistics Features
sales_series['Lag_1'] = sales_series['Sales'].shift(1)
sales_series['Lag_7'] = sales_series['Sales'].shift(7)
sales_series['Rolling_Mean_3'] = sales_series['Sales'].shift(1).rolling(window=3).mean()

# Drop rows with NaN values caused by shifts/rolling windows
sales_series.dropna(inplace=True)

# 3. Chronological Time-based Train/Test Split
X_ts = sales_series[['Lag_1', 'Lag_7', 'Rolling_Mean_3']]
y_ts = sales_series['Sales']

# Retain the final 14 steps cleanly for validation (No random shuffling!)
split_idx = len(sales_series) - 14
X_train_ts, X_test_ts = X_ts.iloc[:split_idx], X_ts.iloc[split_idx:]
y_train_ts, y_test_ts = y_ts.iloc[:split_idx], y_ts.iloc[split_idx:]

# 4. Fit Linear Autoregressive Forecasting Model
ts_model = LinearRegression().fit(X_train_ts, y_train_ts)
ts_preds = ts_model.predict(X_test_ts)

print("--- Sequential Time Series Metrics ---")
print(f"Forecast MAE:  {mean_absolute_error(y_test_ts, ts_preds):.4f}")
print(f"Forecast RMSE: {np.sqrt(mean_squared_error(y_test_ts, ts_preds)):.4f}")

## 4. Random Forest Classifier for Credit Risk/Scoring
**Dataset Target:** Tabular Non-linear Dataset (Credit/Risk Default Profiling)

### Study Checklist
- [ ] Bagging & Bootstrap Sampling Properties
- [ ] Feature Randomness Tuning (`max_features` limits)
- [ ] Tree Ensemble Depth and Node Splitting Criteria
- [ ] Evaluation Metrics via Classification Report
- [ ] MDI Gini vs. Permutation Feature Importances

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# 1. Generate Synthetic Credit Tabular Dataset
X_raw, y_raw = make_classification(
    n_samples=1200, n_features=10, n_informative=6, n_redundant=4, 
    weights=[0.85, 0.15], random_state=RANDOM_SEED
)

df_rf = pd.DataFrame(X_raw, columns=[f"Feature_{i}" for i in range(10)])

# 2. Train/Test Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    df_rf, y_raw, test_size=0.25, stratify=y_raw, random_state=RANDOM_SEED
)

# 3. Production Ensemble Pipeline
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=150, 
        max_depth=6, 
        class_weight='balanced', 
        random_state=RANDOM_SEED
    ))
])

# 4. Execution
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print("--- Random Forest Credit Risk Evaluation ---")
print(classification_report(y_test, y_pred))

# 5. Extrapolate Global Feature Importance Measurements
importances = rf_pipeline.named_steps['model'].feature_importances_
for idx, val in enumerate(importances[:3]):
    print(f"Feature_{idx} Global Gini Contribution: {val:.4f}")

## 5. Support Vector Machine (SVM) + PCA for High-Dimensional Data
**Dataset Target:** MNIST Handwritten Digits (High Dimensional Feature Matrices)

### Study Checklist
- [ ] Multi-dimensional Spatial Feature Layout
- [ ] Principal Component Analysis Covariance & Variance Reconstruction
- [ ] Maximizing Margin Planes via Support Vectors
- [ ] Evaluating RBF vs. Linear Hyperplane Kernels
- [ ] Hyperparameter Tuning Optimization via `C` & `gamma`

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: SVM + PCA
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

# 1. Load High-Dimensional Digits Dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, stratify=y_digits, random_state=RANDOM_SEED
)

# 2. Build Pipeline (Scale -> PCA Compress -> Non-Linear RBF Kernel SVM)
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=RANDOM_SEED)), # Keep 95% variance explained
    ('model', SVC(kernel='rbf', C=2.0, gamma='scale', random_state=RANDOM_SEED))
])

# 3. Execution
svm_pipeline.fit(X_train, y_train)
y_pred = svm_pipeline.predict(X_test)

print("--- High-Dimensional PCA + SVM Metrics ---")
print(f"Accuracy Metric Snapshot: {accuracy_score(y_test, y_pred):.4f}")
print(f"Retained PCA Components Count: {svm_pipeline.named_steps['pca'].n_components_}")

## 6. K-Means Clustering for Customer Segmentation
**Dataset Target:** Unsupervised Clustering (Feature Cohort Optimization)

### Study Checklist
- [ ] Centroid Allocation Optimization Loops
- [ ] Absolute Euclidean Feature Scaling Dependencies
- [ ] Elbow Curves Optimization via Inertia Scoring
- [ ] Silhouette Score Cohesion Evaluation
- [ ] Post-Clustering Distribution Profiling

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Unsupervised K-Means Segmentation
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# 1. Generate Synthetic Segmentation Blobs
X_blob, _ = make_blobs(n_samples=800, n_features=4, centers=3, random_state=RANDOM_SEED)
df_cluster = pd.DataFrame(X_blob, columns=['Spend_Score', 'Activity_Index', 'Feature_3', 'Feature_4'])

# 2. Standardize Features for Equal Distance Calculations
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_cluster)

# 3. Build K-Means Clustering Model (Iterative Centroid Tuning)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED)
cluster_assignments = kmeans.fit_predict(scaled_features)

# 4. Evaluation Profiles
sil_avg = silhouette_score(scaled_features, cluster_assignments)
df_cluster['Cluster_Labels'] = cluster_assignments

print("--- Unsupervised Segment Allocations ---")
print(f"Within-Cluster Inertia: {kmeans.inertia_:.4f}")
print(f"Average Silhouette Cohesion Coefficient: {sil_avg:.4f}")
print("\nCluster Center Map (Scaled Coordinates):")
print(kmeans.cluster_centers_)

## 7. Multinomial Naive Bayes for Text Classification
**Dataset Target:** Tokenized Text Matrix Structures (NLP Multi-Class Target Tasks)

### Study Checklist
- [ ] Prior Probabilities Allocation Models
- [ ] Term Frequency-Inverse Document Frequency (TF-IDF) Calculations
- [ ] Conditional Feature Independence Assumptions
- [ ] Laplace Smoothing Parametric Transformations (`alpha` adjustments)
- [ ] Multi-class Precision, Recall, and Confusion Matrices

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# Practical Implementation: Multinomial Naive Bayes
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups

# 1. Fetch Text Classification Benchmark Categories
categories = ['sci.space', 'comp.graphics', 'rec.sport.hockey']
news_train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
news_test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

# 2. Setup Vectorizer -> Naive Bayes Processing Pipeline
nlp_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, stop_words='english')),
    ('model', MultinomialNB(alpha=0.5)) # Implementing Laplace smoothing parameter
])

# 3. Fit and Evaluate Performance Metrics
nlp_pipeline.fit(news_train.data, news_train.target)
y_pred = nlp_pipeline.predict(news_test.data)

print("--- NLP Multinomial Naive Bayes Metrics ---")
print(classification_report(news_test.target, y_pred, target_names=news_train.target_names))
print("Confusion Matrix Output:")
print(confusion_matrix(news_test.target, y_pred))

## Expanded Mini Project Tracker

| Project Name | Architecture Structure | Feature Mapping Framework | Baseline Metric Evaluation | Status |
|---|---|---|---|---|
| **House Pricing** | `Pipeline(Scaler, Ridge)` | Continuous Public Arrays | RMSE / R^2 Evaluation | Not Started |
| **Customer Churn** | `Pipeline(ColumnTransformer, LogReg)` | Mixed Categorical Matrix | F1 Classification Target Validation | Not Started |
| **Sales Forecasting** | `LinearRegression(Lags, Rolling)` | Time-Series Extrapolations | Mean Absolute Error Analysis | Not Started |
| **Credit Scoring** | `Pipeline(Scaler, RandomForest)` | Weighted Tabular Blobs | Class Validation Precision Map | Not Started |
| **MNIST Dimensionality**| `Pipeline(Scaler, PCA, SVC)` | Multi-Dimensional Compressed Array| High Dimensional Accuracy Metric | Not Started |
| **Customer Cohorts** | `Pipeline(Scaler, KMeans)` | Unsupervised Scaled Spacial Nodes | Silhouette Cluster Verification | Not Started |
| **Text Classification** | `Pipeline(TfidfVectorizer, MultNB)`| Sparse String Token Matrix | Sparse Token Multi-class F1-Score | Not Started |

## Expanded Experiment Log

| Experiment Timestamp | Context Target | Model Architecture Base | Parametric Configuration | validation Metrics Metric | Current State Evaluation |
|---|---|---|---|---|---|
| YYYY-MM-DD | MNIST Compression | PCA + SVC | components=0.95, C=2.0 | Multi-class Accuracy Check | Iteration Baseline Logged |